# ATM Machine Simulation Using OOP
## Overview
This notebook implements a simple ATM machine simulation using Python's object-oriented programming. The system includes:
- A `BankAccount` class to manage account details and transactions.
- A `User` class to represent the account holder.
- An `ATM` class to handle user interactions like checking balance, depositing, withdrawing, and transferring funds.
- The code emphasizes OOP concepts such as encapsulation, abstraction, and class relationships.
- Input validation and error handling are included for a realistic simulation.

In [1]:
# Import datetime for transaction timestamps
import datetime

### Define the `BankAccount` Class
The `BankAccount` class represents a bank account with attributes like account number, balance, and transaction history. Methods include depositing, withdrawing, transferring funds, and retrieving the balance and history. Encapsulation is used by making attributes private (using underscores).

In [2]:
# Define the BankAccount class to manage account details
class BankAccount:
    # Initialize the account with an account number, user, and initial balance
    def __init__(self, account_number, user, initial_balance=0):
        # Private attribute for account number
        self._account_number = account_number
        # Private attribute for user (account holder)
        self._user = user
        # Private attribute for balance
        self._balance = initial_balance
        # Private list to store transaction history
        self._transactions = []
        # Log the initial balance as a transaction
        self._add_transaction("Initial deposit", initial_balance)

    # Private method to log transactions with timestamp
    def _add_transaction(self, transaction_type, amount):
        # Create a transaction dictionary with type, amount, and timestamp
        transaction = {
            "type": transaction_type,
            "amount": amount,
            "timestamp": datetime.datetime.now()
        }
        # Append the transaction to the history
        self._transactions.append(transaction)

    # Method to deposit money into the account
    def deposit(self, amount):
        # Check if the deposit amount is positive
        if amount > 0:
            # Add the amount to the balance
            self._balance += amount
            # Log the deposit transaction
            self._add_transaction("Deposit", amount)
            # Return True to indicate success
            return True
        # Return False if the amount is invalid
        return False

    # Method to withdraw money from the account
    def withdraw(self, amount):
        # Check if the amount is positive and sufficient funds are available
        if amount > 0 and self._balance >= amount:
            # Subtract the amount from the balance
            self._balance -= amount
            # Log the withdrawal transaction
            self._add_transaction("Withdrawal", -amount)
            # Return True to indicate success
            return True
        # Return False if the amount is invalid or insufficient funds
        return False

    # Method to transfer money to another account
    def transfer(self, recipient_account, amount):
        # Check if the amount is positive and sufficient funds are available
        if amount > 0 and self._balance >= amount:
            # Subtract the amount from this account
            self._balance -= amount
            # Add the amount to the recipient's account
            recipient_account._balance += amount
            # Log the transfer in this account
            self._add_transaction(f"Transfer to {recipient_account._account_number}", -amount)
            # Log the transfer in the recipient's account
            recipient_account._add_transaction(f"Transfer from {self._account_number}", amount)
            # Return True to indicate success
            return True
        # Return False if the amount is invalid or insufficient funds
        return False

    # Method to get the current balance
    def get_balance(self):
        # Return the current balance
        return self._balance

    # Method to get the transaction history
    def get_transaction_history(self):
        # Return the list of transactions
        return self._transactions


### Define the `User` Class
The `User` class represents the account holder with attributes like name and PIN. The PIN is used for authentication during ATM operations. This class demonstrates encapsulation by keeping the PIN private.


In [3]:
# Define the User class to represent the account holder
class User:
    # Initialize the user with a name and PIN
    def __init__(self, name, pin):
        # Public attribute for name
        self.name = name
        # Private attribute for PIN
        self._pin = pin

    # Method to verify the PIN
    def verify_pin(self, pin):
        # Return True if the provided PIN matches the stored PIN
        return self._pin == pin


### Define the `ATM` Class
The `ATM` class manages user interactions, such as authenticating the user, displaying the menu, and performing transactions. It interacts with the `BankAccount` and `User` classes. This class abstracts the ATM's functionality, hiding implementation details from the user.


In [4]:
# Define the ATM class to handle user interactions
class ATM:
    # Initialize the ATM with a bank account
    def __init__(self, bank_account):
        # Store the bank account
        self._bank_account = bank_account
        # Track whether a user is authenticated
        self._authenticated = False

    # Method to authenticate the user
    def authenticate(self, user, pin):
        # Verify the user's PIN
        if user.verify_pin(pin):
            # Set authenticated to True
            self._authenticated = True
            # Return True to indicate successful authentication
            return True
        # Return False if authentication fails
        return False

    # Method to display the ATM menu and handle user choices
    def run(self):
        # Check if the user is authenticated
        if not self._authenticated:
            print("Please authenticate first.")
            return

        # Loop to display the menu until the user exits
        while True:
            # Display the menu options
            print("\nATM Menu:")
            print("1. Check Balance")
            print("2. Deposit")
            print("3. Withdraw")
            print("4. Transfer")
            print("5. Transaction History")
            print("6. Exit")
            # Get the user's choice
            choice = input("Enter choice (1-6): ")

            # Handle the user's choice
            if choice == "1":
                # Get and display the current balance
                balance = self._bank_account.get_balance()
                print(f"Current Balance: ${balance:.2f}")

            elif choice == "2":
                # Get the deposit amount from the user
                try:
                    amount = float(input("Enter amount to deposit: $"))
                    # Attempt to deposit the amount
                    if self._bank_account.deposit(amount):
                        print(f"Successfully deposited ${amount:.2f}")
                    else:
                        print("Invalid deposit amount.")
                except ValueError:
                    print("Please enter a valid number.")

            elif choice == "3":
                # Get the withdrawal amount from the user
                try:
                    amount = float(input("Enter amount to withdraw: $"))
                    # Attempt to withdraw the amount
                    if self._bank_account.withdraw(amount):
                        print(f"Successfully withdrew ${amount:.2f}")
                    else:
                        print("Invalid amount or insufficient funds.")
                except ValueError:
                    print("Please enter a valid number.")

            elif choice == "4":
                # Get the recipient's account number and transfer amount
                recipient_account_number = input("Enter recipient account number: ")
                try:
                    amount = float(input("Enter amount to transfer: $"))
                    # Note: For simplicity, create a dummy recipient account
                    # In a real system, this would be retrieved from a database
                    recipient_user = User("Recipient", "0000")
                    recipient_account = BankAccount(recipient_account_number, recipient_user)
                    # Attempt to transfer the amount
                    if self._bank_account.transfer(recipient_account, amount):
                        print(f"Successfully transferred ${amount:.2f} to {recipient_account_number}")
                    else:
                        print("Invalid amount or insufficient funds.")
                except ValueError:
                    print("Please enter a valid number.")

            elif choice == "5":
                # Get and display the transaction history
                transactions = self._bank_account.get_transaction_history()
                print("\nTransaction History:")
                for transaction in transactions:
                    print(f"{transaction['timestamp']}: {transaction['type']} ${abs(transaction['amount']):.2f}")

            elif choice == "6":
                # Exit the menu
                print("Thank you for using the ATM. Goodbye!")
                break

            else:
                # Handle invalid menu choices
                print("Invalid choice. Please try again.")


### Simulate the ATM Usage
We create a user, a bank account, and an ATM instance. We then authenticate the user and run the ATM interface. This simulates a real-world scenario where a user interacts with the ATM.


In [5]:
# Create a user with name and PIN
user = User("John Doe", "1234")
# Create a bank account with an account number, user, and initial balance
account = BankAccount("123456789", user, 1000)
# Create an ATM instance with the bank account
atm = ATM(account)

# Authenticate the user with the correct PIN
if atm.authenticate(user, "1234"):
    print("Authentication successful!")
    # Run the ATM interface
    atm.run()
else:
    print("Authentication failed.")


Authentication successful!

ATM Menu:
1. Check Balance
2. Deposit
3. Withdraw
4. Transfer
5. Transaction History
6. Exit
Enter choice (1-6): 1
Current Balance: $1000.00

ATM Menu:
1. Check Balance
2. Deposit
3. Withdraw
4. Transfer
5. Transaction History
6. Exit
Enter choice (1-6): 2
Enter amount to deposit: $1000
Successfully deposited $1000.00

ATM Menu:
1. Check Balance
2. Deposit
3. Withdraw
4. Transfer
5. Transaction History
6. Exit
Enter choice (1-6): 2
Enter amount to deposit: $1000
Successfully deposited $1000.00

ATM Menu:
1. Check Balance
2. Deposit
3. Withdraw
4. Transfer
5. Transaction History
6. Exit
Enter choice (1-6): 1
Current Balance: $3000.00

ATM Menu:
1. Check Balance
2. Deposit
3. Withdraw
4. Transfer
5. Transaction History
6. Exit
Enter choice (1-6): 3
Enter amount to withdraw: $1000
Successfully withdrew $1000.00

ATM Menu:
1. Check Balance
2. Deposit
3. Withdraw
4. Transfer
5. Transaction History
6. Exit
Enter choice (1-6): 1
Current Balance: $2000.00

ATM Menu:

## Explanation of OOP Concepts Used
1. **Encapsulation**: Attributes like `_balance`, `_pin`, and `_transactions` are private (indicated by the underscore), accessible only through methods. This protects the data from external modification.
2. **Abstraction**: The `ATM` class hides the complexity of transactions and authentication, providing a simple interface (menu) for the user.
3. **Class Relationships**: The `ATM` class interacts with `BankAccount` and `User` classes, demonstrating composition (ATM contains a reference to a `BankAccount`).
4. **Modularity**: Each class has a single responsibility (`User` for authentication, `BankAccount` for transactions, `ATM` for user interaction), making the code reusable and maintainable.

## How to Run
1. Copy the code into a Jupyter Notebook, separating markdown and code cells as shown.
2. Run each cell in sequence.
3. After running the final cell, interact with the ATM by entering menu choices and amounts as prompted.
4. The program will simulate ATM operations like checking balance, depositing, withdrawing, transferring, and viewing transaction history.

## Notes
- The transfer functionality uses a dummy recipient account for simplicity. In a real system, recipient accounts would be retrieved from a database.
- Error handling ensures invalid inputs (e.g., negative amounts or non-numeric inputs) are caught gracefully.
- Transaction history includes timestamps for realism.
- The code is kept simple but can be extended with features like multiple accounts, PIN retries, or a database.
